In [2]:
# !pip install transformers peft datasets accelerate

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import torch

model_name = "distilgpt2"  # ✅ 82MB, CPU-friendly, no auth needed

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

# GPT-2 uses c_attn for attention layers
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn"],  # GPT-2 specific
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # expect ~0.5% trainable

# Your training data
data = [
    {"text": "Question: What is 2+2? Answer: 4"},
    {"text": "Question: Capital of France? Answer: Paris"},
    {"text": "Question: Who wrote Hamlet? Answer: Shakespeare"},
    {"text": "Question: What color is the sky? Answer: Blue"},
]
dataset = Dataset.from_list(data)

def tokenize(batch):
    tokens = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=64  # short = faster on CPU
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

training_args = TrainingArguments(
    output_dir="./lora-output",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    learning_rate=2e-4,
    fp16=False,         # ✅ CPU safe
    bf16=False,         # ✅ CPU safe
    logging_steps=1,
    save_strategy="no", # skip saving checkpoints mid-training
    report_to="none",   # no wandb
    use_cpu=True,       # explicit CPU mode
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()

model.save_pretrained("./lora-adapters")
tokenizer.save_pretrained("./lora-adapters")
print("✅ Done! Adapters saved to ./lora-adapters")

/Users/shraddhasharma/Desktop/pp/Agentic AI Workflow/fine tuning/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 76/76 [00:00<00:00, 10117.02it/s]
/Users/shraddhasharma/Desktop/pp/Agentic AI Workflow/fine tuning/.venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


Map: 100%|██████████| 4/4 [00:00<00:00, 255.35 examples/s]
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,9.491064
2,9.543443
3,9.501981
4,10.378103
5,8.800491
6,9.819681
7,9.622227
8,9.168859
9,9.450048
10,9.503306


✅ Done! Adapters saved to ./lora-adapters
